In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syts/all"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "unfolding-fake_data_tests-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Plot Syts

In [ ]:
var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

In [ ]:
var_config = VariableConfig.muon_momentum()

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"

mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
flux_syst = np.load(file_dir + "/flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/g4_syst_dict_xsec.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
genie_syst = np.load(file_dir + "/genie_syst_dict_xsec.npz")

detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")

# flat uncertainties
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01
nu_score= 0.03

In [ ]:
for var_config in var_configs:
    fig, ax = plt.subplots()
    
    frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
    systs      = [mcstat_syst, flux_syst, g4_syst, genie_syst,  detvar_syst, cosmics_syst]
    syst_names = ["MCStat", "Flux", "G4", "Genie", "Detector", "Cosmic"]
    
    # flat systs
    flat_systs = [pot_frac_unc, ntargets_frac_unc]
    flat_syst_names = ["POT", "Ntargets"]
    
  # First group (should appear on top)
    for syst_name, syst in zip(syst_names, systs):
        syst_uncert = np.sqrt(np.diag(syst[var_config.var_save_name]))
        frac_uncert_total += syst_uncert ** 2
    
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=3
        )
    
    # Second group (should appear underneath)
    for syst_name, syst in zip(flat_syst_names, flat_systs):
        syst_uncert = syst * np.ones(len(var_config.bin_centers))
        frac_uncert_total += syst_uncert ** 2
    
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=1
        )
    
    frac_uncert_total = np.sqrt(frac_uncert_total)
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total * 1e2,    histtype="step", linewidth=2, color="k",  label="Total")
    
    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.ylim(0, max(frac_uncert_total*1e2) * 1.4)
    
    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Uncertainty [%]")
    plt.legend(fontsize=11, ncol=3, loc="upper center")
    
    plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    plt.minorticks_on()

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"

mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
flux_syst = np.load(file_dir + "/flux_syst_dict_rate.npz")
g4_syst = np.load(file_dir + "/g4_syst_dict_rate.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
genie_syst = np.load(file_dir + "/genie_syst_dict_rate.npz")

detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")

# flat uncertainties
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01
nu_score= 0.03

In [ ]:
for var_config in var_configs:
    fig, ax = plt.subplots()
    
    frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
    systs      = [mcstat_syst, flux_syst, g4_syst, genie_syst,  detvar_syst, cosmics_syst]
    syst_names = ["MCStat", "Flux", "G4", "Genie", "Detector", "Cosmic"]
    
    # flat systs
    flat_systs = [pot_frac_unc, ntargets_frac_unc,nu_score]
    flat_syst_names = ["POT", "Ntargets", "NuScore"]
    
    # First group (should appear on top)
    for syst_name, syst in zip(syst_names, systs):
        syst_uncert = np.sqrt(np.diag(syst[var_config.var_save_name]))
        frac_uncert_total += syst_uncert ** 2
    
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=3
        )
    
    # Second group (should appear underneath)
    for syst_name, syst in zip(flat_syst_names, flat_systs):
        syst_uncert = syst * np.ones(len(var_config.bin_centers))
        frac_uncert_total += syst_uncert ** 2
    
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=1
        )
    
    frac_uncert_total = np.sqrt(frac_uncert_total)
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total * 1e2,    histtype="step", linewidth=2, color="k",  label="Total")
    
    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.ylim(0, max(frac_uncert_total*1e2) * 1.4)
    
    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Uncertainty [%]")
    plt.legend(fontsize=11, ncol=3, loc="upper center")
    
    plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    plt.minorticks_on()